In [1]:
from time import time

import pandas as pd
from tabulate import tabulate

import torch
import torch.nn as nn

from linear_with_fp8_cast import Fp8Linear
from training_utils import prepare_mnist_dataset_for_training, train_model, test_model

/home/msst/Utils/miniconda3/envs/qenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIM = 768
BATCH_SIZE = 8192
HIDDEN_DIM = 8192
NUM_CLASSES = 10

In [3]:
# Prepare data
train_loader, test_loader = prepare_mnist_dataset_for_training(BATCH_SIZE)

In [4]:
class SimpleMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        
        self.act = nn.ReLU()
        self.linear1 = torch.nn.Linear(self.in_dim, self.hidden_dim, bias=False)
        self.linear2 = torch.nn.Linear(self.hidden_dim, self.hidden_dim, bias=False)
        self.linear3 = torch.nn.Linear(self.hidden_dim, self.hidden_dim, bias=False)
        self.head = torch.nn.Linear(self.hidden_dim, self.out_dim, bias=False)
        
    def forward(self, x):
        x = self.linear1(x)
        x = self.act(x)
        x = self.linear2(x)
        x = self.act(x)
        x = self.linear3(x)
        x = self.act(x)
        x = self.head(x)
        return x
    
    @torch.no_grad()
    def set_fp8_layers(self):
        for l_name in ["linear1", "linear2", "linear3",]:
            original_module = self.get_submodule(l_name)
            new_module = Fp8Linear(in_features=original_module.in_features, out_features=original_module.out_features, bias=False)
            new_module.weight.data.copy_(original_module.weight.data)
            new_module = new_module.to(original_module.weight.device)
            setattr(self, l_name, new_module)

In [5]:
# Training_setting

configs = [
    "fp32", 
    "fp32_autocast_bf16",
    "bf16",
    "bf16_with_fp8_kernel"
]

def get_setting_variables(config):
    if config == "fp32":
        weight_dtype = torch.float32
        autocast_dtype = None
        use_fp8_linear = False
    elif config == "fp32_autocast_bf16":
        weight_dtype = torch.float32
        autocast_dtype = torch.bfloat16 
        use_fp8_linear = False
    elif config == "bf16":
        weight_dtype = torch.bfloat16
        autocast_dtype = None
        use_fp8_linear = False
    elif config == "bf16_with_fp8_kernel":
        weight_dtype = torch.bfloat16
        autocast_dtype = None
        use_fp8_linear = True
    return weight_dtype, autocast_dtype, use_fp8_linear

In [6]:
results = []

for config in configs:
    print("run config:", config)
    weight_dtype, autocast_dtype, use_fp8_linear = get_setting_variables(config)

    # Move data to gpu to reduce loading bottleneck
    fast_train_loader = [[
        batch['image'].reshape(BATCH_SIZE, DATA_DIM).to(weight_dtype).cuda(),
        batch['label'].cuda()
    ] for batch in train_loader]

    fast_test_loader = [[
        batch['image'].reshape(BATCH_SIZE, DATA_DIM).to(weight_dtype).cuda(),
        batch['label'].cuda()
    ] for batch in test_loader]

    # Prepare model and optimizer
    model = SimpleMLP(in_dim=DATA_DIM, hidden_dim=HIDDEN_DIM, out_dim=NUM_CLASSES).to(weight_dtype).cuda()

    if use_fp8_linear:
        model.set_fp8_layers()

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Warmup_epoch
    train_model(
        model=model,
        optimizer=optimizer,
        train_loader=fast_train_loader,
        loss_fn=loss_fn, 
        num_epochs=1, 
        autocast_dtype=autocast_dtype
    )

    t0 = time()
    avg_loss = train_model(
        model=model,
        optimizer=optimizer,
        train_loader=fast_train_loader,
        loss_fn=loss_fn, 
        num_epochs=10,
        autocast_dtype=autocast_dtype
    )
    t1 = time()
    training_time = t1-t0

    test_accuracy = test_model(
        model=model, 
        test_loader=fast_test_loader
    )

    del model, optimizer
    torch.cuda.empty_cache()

    results.append({
            'Config': config,
            'Loss': f'{avg_loss:.4f}',
            'Test Accuracy': f'{test_accuracy:.2f}%',
            'Training Time': f'{training_time:.2f}s'
        })

run config: fp32


KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(results)
print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))

+----------------------+--------+-----------------+-----------------+
| Config               |   Loss | Test Accuracy   | Training Time   |
+======================+========+=================+=================+
| fp32                 | 0.0271 | 97.83%          | 17.52s          |
+----------------------+--------+-----------------+-----------------+
| fp32_autocast_bf16   | 0.0302 | 97.66%          | 6.78s           |
+----------------------+--------+-----------------+-----------------+
| bf16                 | 0.0244 | 97.79%          | 6.13s           |
+----------------------+--------+-----------------+-----------------+
| bf16_with_fp8_kernel | 0.0226 | 97.64%          | 5.32s           |
+----------------------+--------+-----------------+-----------------+
